## Diaformer on DDXPlus

This test uses a limited set of patients, all questions and answers will be converted to yes/no to match the style of diaformer

### import and split

In [7]:
from pathlib import Path
import ast
import pickle
import pandas as pd
import ast
import string

test = "initial"
trainNo = 2000
validNo = 100
testNo = 100
epochs = 1
batch_size = 8
max_turns = 20
rndseed = 10

data = Path("C:\\Users\\Hzaab\\SDP-Sequential-Diagnosis\\data")
prepData= Path("C:\\Users\\Hzaab\\SDP-Sequential-Diagnosis\\diaformer\\Data")
source = Path("C:\\Users\\Hzaab\\SDP-Sequential-Diagnosis\\diaformer\\Diaformer_source")
output = Path("C:\\Users\\Hzaab\\SDP-Sequential-Diagnosis\\outputs\\" + test)

In [8]:
evidences = pd.read_json("C:\\Users\\Hzaab\\SDP-Sequential-Diagnosis\\data\\release_evidences.json", orient="index")

conditions = pd.read_json("C:\\Users\\Hzaab\\SDP-Sequential-Diagnosis\\data\\release_conditions.json", orient="index")

print("evidences:", len(evidences), ", conditions:", len(conditions))

evidences: 223 , conditions: 49


In [9]:
columns = ["PATHOLOGY", "EVIDENCES", "INITIAL_EVIDENCE"]

train_df = pd.read_csv("C:\\Users\\Hzaab\\SDP-Sequential-Diagnosis\\data\\release_test_patients\\release_test_patients.csv", usecols=columns)
validation_df = pd.read_csv("C:\\Users\\Hzaab\\SDP-Sequential-Diagnosis\\data\\release_validate_patients\\release_validate_patients.csv", usecols=columns)
test_df = pd.read_csv("C:\\Users\\Hzaab\\SDP-Sequential-Diagnosis\\data\\release_test_patients\\release_test_patients.csv", usecols=columns)

train_df = train_df.sample(n=min(trainNo, len(train_df)), random_state=rndseed)
validation_df = validation_df.sample(n=min(validNo, len(validation_df)), random_state=rndseed)
test_df = test_df.sample(n=min(testNo, len(test_df)), random_state=rndseed)

print("Train:", len(train_df), "Validation:", len(validation_df), "Test:", len(test_df))
train_df.head(3)

Train: 2000 Validation: 100 Test: 100


,PATHOLOGY,EVIDENCES,INITIAL_EVIDENCE
116748,Possible NSTEMI / STEMI,"['E_2', 'E_53', 'E_54_@_V_183', 'E_54_@_V_193'...",E_53
12410,Acute dystonic reactions,"['E_15', 'E_62', 'E_66', 'E_128', 'E_147', 'E_...",E_66
96034,Chronic rhinosinusitis,"['E_53', 'E_54_@_V_181', 'E_54_@_V_192', 'E_55...",E_103


### convert data to input for diaformer

for every row, remove the initial evidence from the evidences, then save the patient info in the format that diaformer requires

In [ ]:

reformat = []

for frame in [train_df, validation_df, test_df]:
    patients = []

    for index, row in frame.iterrows():

        patient_evidences = ast.literal_eval(row["EVIDENCES"])
        initial = row["INITIAL_EVIDENCE"]

        hidden = [evidence for evidence in patient_evidences if evidence != initial]

        patients.append({
            "disease_tag": row.PATHOLOGY,
            "goal": {
                "explicit_inform_slots": dict.fromkeys([initial], True),
                "implicit_inform_slots": dict.fromkeys(hidden, True),
            },
        })

    reformat.append(patients)

train_patients, validation_patients, test_patients = reformat

train_patients[0]

In [10]:
reformat = []

for frame in [train_df, validation_df, test_df]:
    patients = []

    for index, row in frame.iterrows():

        patient_evidences = ast.literal_eval(row["EVIDENCES"])
        initial = row["INITIAL_EVIDENCE"]

        if initial not in patient_evidences:
            continue

        hidden = [evidence for evidence in patient_evidences if evidence != initial]

        patients.append({
            "disease_tag": row["PATHOLOGY"],
            "goal": {
                "explicit_inform_slots": {initial: True},
                "implicit_inform_slots": {evidence: True for evidence in hidden},
            },
        })

    reformat.append(patients)

train_patients, validation_patients, test_patients = reformat

train_patients[0]

{'disease_tag': 'Possible NSTEMI / STEMI',
 'goal': {'explicit_inform_slots': {'E_53': True},
  'implicit_inform_slots': {'E_2': True,
   'E_54_@_V_183': True,
   'E_54_@_V_193': True,
   'E_54_@_V_196': True,
   'E_55_@_V_33': True,
   'E_55_@_V_55': True,
   'E_55_@_V_159': True,
   'E_55_@_V_195': True,
   'E_55_@_V_197': True,
   'E_56_@_10': True,
   'E_57_@_V_30': True,
   'E_57_@_V_31': True,
   'E_57_@_V_39': True,
   'E_57_@_V_163': True,
   'E_57_@_V_194': True,
   'E_58_@_2': True,
   'E_59_@_10': True,
   'E_69': True,
   'E_71': True,
   'E_79': True,
   'E_89': True,
   'E_105': True,
   'E_108': True,
   'E_148': True,
   'E_191': True,
   'E_204_@_V_10': True,
   'E_225': True}}}

In [12]:

with (prepData / "goal_set.p").open("wb") as file:
    pickle.dump({"train": train_patients, "test": validation_patients}, file) #naming validation test to match the diaformer code, it will be used as validation

with (prepData / "test_goal_set.p").open("wb") as file:
    pickle.dump({"test": test_patients}, file)


diaformer needs a list of all evidences and diagnoses

In [13]:
all_evidences = []

for index, evidence in evidences.iterrows():
    name = evidence["name"]

    if evidence["data_type"] == "B": #B data type is the True/False evidences
        all_evidences.append(name)

    else: #the other one has multiple values, so diafromer needs a token for each possible value
        for value in evidence["possible-values"]:
            all_evidences.append(f"{name}_@_{value}")

symptoms = sorted(set(all_evidences))
diseases = sorted(conditions["condition_name"].unique())

#diaformer requires these special tokens in this order
special = ["[PAD]", "[PAD2]", "[UNK]", "[SEP]","[CLS]", "[MASK]", "[true]", "[false]"]

with open(prepData / "vocab.txt", "w", encoding="utf-8") as file:
    for token in special + symptoms:
        file.write(token + "\n")

    file.write("\n")

    for disease in diseases:
        file.write(disease + "\n")

print("evidence tokens:", len(symptoms))
print("diseases:", len(diseases))

evidence tokens: 972
diseases: 49
